# Exploration/dev notebook for scheduling and bridge to scheduling from treatments

In [1]:
import os
import pandas as pd
from pathlib import Path
import sqlalchemy as sa
import sqlalchemy.orm as so
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

In [2]:
REGISTRY_PATH = ROOT / "scoop" / "configuration" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"

In [3]:
HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]
def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved
engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

In [4]:
from scoop.configuration.protocol_registry import RegistryYaml
from scoop.utils.scheduling import SystemicPhase
from scoop.utils.treatment_containers import ProtocolGenerator

In [5]:
with so.Session(engine) as session:
    nsclc_registry = RegistryYaml.from_yaml_path(REGISTRY_PATH, session)

In [6]:
nsclc_trt_generator = ProtocolGenerator(source_phases=nsclc_registry.source_phases, phase_unions=nsclc_registry.union_phases)

In [7]:
nsclc_registry.protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], optional_phases=None),
 'neoadjuvant_or_perioperative_ici_w_chemo': ProtocolYaml(protocol_id='neoadjuvant_or_perioperative_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab', source_phase_ids=['conc_crt', 'consolidation_durvalumab'], optional_phases=None),
 'conc_crt_foll

## Getting dates for a systemic phase
(The durvalumab sig bundle has implicit portions, each w/ dif cycle length and is rolled out incorrectly if done with each sig independent from table)

In [8]:
with so.Session(engine) as session:
    neoadj_or_peri = nsclc_trt_generator.get_protocols_from_protocol_yaml(
        nsclc_registry.protocols["neoadjuvant_or_perioperative_ici_w_chemo"], session
        )

In [9]:
neoadj_w_durvalumab = neoadj_or_peri[1].phases[0].options[0]
neoadj_w_durvalumab

Cisplatin and Docetaxel (DC) and Durvalumab Variant #01 (CUI: 130688)
Cisplatin: 100 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4936
Docetaxel: 85 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4937
Durvalumab: 750 mg IV once on day(s) 1 in cycle(s) 4,5. Neoadjuvant SigID: 4938

In [10]:
neoadj_durv_schedule = SystemicPhase.from_component_sigs(neoadj_w_durvalumab.sigs)
neoadj_durv_schedule

SystemicPhase(cycle_sets=[CycleSet(seed=None, cycle_length_lb=21, cycle_length_ub=21, cycle_len_unit='day', cycle_sigs='21-day cycle for 3 cycles', trios=[Trio(per_cycle_days=PerCycleDays(every=[1], base=[1], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Cisplatin', cui=105, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7ff5794ad040>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=100.0, dosemaxnum=100.0, doseunit='mg/m^2', doseunit_cui=134448), duration=None, role='primary systemic'), cycle_events=CycleEvents(every=[1, 2, 3], base=[1, 2, 3], optional=[], ongoing_step=None, first_event=1)), Trio(per_cycle_days=PerCycleDays(every=[1], base=[1], optional=[], ongoing_step=None), component_part=ComponentPart(component=Component(name='Docetaxel', cui=164, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7ff57957a650>),

In [11]:
neoadj_durv_trt_dates = neoadj_durv_schedule.get_scheduled_dates(pd.to_datetime("2015-01-01"))
neoadj_durv_trt_dates

defaultdict(list,
            {Timestamp('2015-01-02 00:00:00'): [ComponentPart(component=Component(name='Cisplatin', cui=105, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7ff5794ad040>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=100.0, dosemaxnum=100.0, doseunit='mg/m^2', doseunit_cui=134448), duration=None, role='primary systemic'),
              ComponentPart(component=Component(name='Docetaxel', cui=164, drug_object=<hemonc_import.final_model.entities.Drugs object at 0x7ff57957a650>), step_int=1, step_max=1, route='IV', route_cui=44957, frequency='once', frequency_cui=139563, dose_info=StandardDoseInfo(seed=None, doseminnum=85.0, dosemaxnum=85.0, doseunit='mg/m^2', doseunit_cui=134448), duration=None, role='primary systemic')],
             Timestamp('2015-01-23 00:00:00'): [ComponentPart(component=Component(name='Cisplatin', cui=105, drug_object=<hemonc_import.f

In [12]:
for date, component_parts in neoadj_durv_trt_dates.items():
    dosing = {component_part.component.name: component_part.get_dose()
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2015-01-02 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-01-23 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-02-13 00:00:00: {'Cisplatin': (100.0, 'mg/m^2'), 'Docetaxel': (85.0, 'mg/m^2')}
2015-03-06 00:00:00: {'Durvalumab': (750.0, 'mg')}
2015-03-20 00:00:00: {'Durvalumab': (750.0, 'mg')}


In [13]:
for date, component_parts in neoadj_durv_trt_dates.items():
    dosing = {component_part.component.name: component_part.get_dose(units_as_cuis=True)
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2015-01-02 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-01-23 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-02-13 00:00:00: {'Cisplatin': (100.0, 134448), 'Docetaxel': (85.0, 134448)}
2015-03-06 00:00:00: {'Durvalumab': (750.0, 134444)}
2015-03-20 00:00:00: {'Durvalumab': (750.0, 134444)}


In [14]:
neoadj_w_nivolumab_eg = neoadj_or_peri[1].phases[0].options[2]
neoadj_w_nivolumab_eg

Cisplatin and Docetaxel (DC) and Nivolumab Variant #02 (CUI: 130690)
Cisplatin: 75 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4942
Docetaxel: 75 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4943
Nivolumab: 360 mg IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4944

In [15]:
neoadj_niv_eg_schedule = SystemicPhase.from_component_sigs(neoadj_w_nivolumab_eg.sigs)

In [16]:
neoadj_niv_eg_dates = neoadj_niv_eg_schedule.get_scheduled_dates(pd.to_datetime("2020-02-05"))

In [17]:
for date, component_parts in neoadj_niv_eg_dates.items():
    dosing = {component_part.component.name: component_part.get_dose()
              for component_part in component_parts}
    print(f"{date}: {dosing}")

2020-02-06 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}
2020-02-27 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}
2020-03-19 00:00:00: {'Cisplatin': (75.0, 'mg/m^2'), 'Docetaxel': (75.0, 'mg/m^2'), 'Nivolumab': (360.0, 'mg')}
